# File Writers

`cimgraph.utils` has two writers that save the graph of a `GraphModel` to a file:

* `write_xml(network, filename)` writes CIM RDF/XML.
* `write_json_ld(network, filename)` writes JSON-LD in the CGMES draft style.

Both write classes in alphabetical order and objects sorted by URI, so the same model gives the same file body on every run. They can be read back with `XMLFile` and `JSONLDFile` (see [XML File Parser](../03_databases/3_8_xml_file_parser.ipynb) and [JSON-LD File Parser](../03_databases/3_9_jsonld_file_parser.ipynb)).

## Load a Model

Load the IEEE 13 bus test feeder from CIM XML:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(connection=XMLFile(filename='../../tests/test_models/ieee13.xml'),
                      container=feeder, distributed=False)

The examples write to a temporary directory and print the start of each file:

In [2]:
import json
import tempfile
from pathlib import Path

out_dir = Path(tempfile.mkdtemp())

def print_head(path, lines=12):
    print(''.join(path.read_text().splitlines(keepends=True)[:lines]))

## CIM XML

`write_xml` writes an `md:FullModel` and `cim:ModelAuthority` header, then every object. Values are plain numbers in base SI units (the line length below is in meters).

In [3]:
from cimgraph.utils import write_xml

xml_file = out_dir / 'ieee13.xml'
write_xml(network, str(xml_file))
print_head(xml_file, lines=19)

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF xmlns:cim="http://iec.ch/TC57/CIM100#" xmlns:md="http://iec.ch/TC57/61970-552/ModelDescription/1#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
<md:FullModel rdf:about="urn:uuid:179bcc33-ef9b-4b82-abb5-4e130ac595ec">
  <md:Model.created>2026-11-21T00:29:47Z</md:Model.created>
  <md:Model.modelingAuthoritySet>https://github.com/PNNL-CIM-Tools/CIM-Graph</md:Model.modelingAuthoritySet>
</md:FullModel>
<cim:ModelAuthority rdf:about="urn:uuid:bdf23f0c-0c08-525b-846a-e05b60f8af02">
  <cim:IdentifiedObject.name>https://github.com/PNNL-CIM-Tools/CIM-Graph</cim:IdentifiedObject.name>
</cim:ModelAuthority>
<cim:ACLineSegment rdf:about="urn:uuid:0BBD0EA3-F665-465B-86FD-FC8B8466AD53">
  <cim:Identity.identifier>0BBD0EA3-F665-465B-86FD-FC8B8466AD53</cim:Identity.identifier>
  <cim:ACLineSegment.PerLengthImpedance rdf:resource="urn:uuid:12C526C5-6645-4C0C-83CD-3030C440DA17"/>
  <cim:Conductor.length>91.44</cim:Conductor.length>
  <cim:Condu

Options:

* `write_identifier=False` leaves out `cim:Identity.identifier`.
* `write_model_header=False` leaves out the header.
* `header={...}` (with `contexts=[...]` for extra prefixes) writes the `dcat:Dataset` header described below instead of `md:FullModel`.

## JSON-LD

`write_json_ld` writes the ENTSO-E metadata layout: the top-level map is a `dcat:Dataset` header and `@graph` holds the objects.

* `@context` is the profile's published context URL.
* References are `"urn:uuid:..."` strings.
* Numbers are strings in the units of a units convention, named by the header's `dct:conformsTo`. The default for `cimhub_2023` is `si`.

In [4]:
from cimgraph.utils import write_json_ld

jsonld_file = out_dir / 'ieee13.jsonld'
write_json_ld(network, str(jsonld_file))

document = json.loads(jsonld_file.read_text())
objects = document.pop('@graph')
print(json.dumps(document, indent=2))
print(json.dumps(objects[0], indent=2))

{
  "@context": "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/context.jsonld",
  "@id": "urn:uuid:31b69c37-9a67-423f-ae2b-2f56ae272344",
  "@type": [
    "dcat:Dataset",
    "http://www.w3.org/2004/03/trix/rdfg-1/Graph"
  ],
  "dct:identifier": "31b69c37-9a67-423f-ae2b-2f56ae272344",
  "dct:issued": "2026-11-21T00:29:47Z",
  "md:Model.modelingAuthoritySet": "https://github.com/PNNL-CIM-Tools/CIM-Graph",
  "dct:conformsTo": [
    "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-si.jsonld"
  ],
  "dcat:distribution": {
    "@type": "dcat:Distribution",
    "dct:conformsTo": [
      "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-si.jsonld"
    ]
  }
}
{
  "@id": "urn:uuid:0BBD0EA3-F665-465B-86FD-FC8B8466AD53",
  "@type": "cim:ACLineSegment",
  "cim:ACLineSegment.PerLengthImpedance": "urn:uuid:12C526C5-6645-4C0C-83CD-3030C440DA17",
  "cim:Conductor.length": "91.44",
  "cim:ConductingEquipment.BaseVoltage": "urn:uuid:2A158E0C-

### Header and Units Convention

By default, `@id`, `dct:identifier` and `dct:issued` are a new UUID and the export time. With `header=`, set them yourself, so identical exports are identical files, and add properties such as `dcat:isVersionOf` (the model series IRI). A header prefix that the contexts do not define raises a `ValueError`; pass `contexts=[{'prefix': 'namespace'}]` to define it.

`convention='distribution'` writes kW, kVAr and kVA, line constants per km and wire sizes in mm:

In [5]:
header = {
    '@id': 'urn:uuid:5a0c3a4e-0000-5000-8000-000000000001',
    'dct:identifier': '5a0c3a4e-0000-5000-8000-000000000001',
    'dct:issued': '2026-11-01T00:00:00Z',
    'dcat:isVersionOf': 'https://example.org/models/ieee13',
}
kw_file = out_dir / 'ieee13_kw.jsonld'
write_json_ld(network, str(kw_file), convention='distribution', header=header)

document = json.loads(kw_file.read_text())
objects = document.pop('@graph')
print(json.dumps(document, indent=2))

{
  "@context": "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/context.jsonld",
  "@id": "urn:uuid:5a0c3a4e-0000-5000-8000-000000000001",
  "@type": [
    "dcat:Dataset",
    "http://www.w3.org/2004/03/trix/rdfg-1/Graph"
  ],
  "dct:identifier": "5a0c3a4e-0000-5000-8000-000000000001",
  "dct:issued": "2026-11-01T00:00:00Z",
  "md:Model.modelingAuthoritySet": "https://github.com/PNNL-CIM-Tools/CIM-Graph",
  "dct:conformsTo": [
    "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-distribution.jsonld"
  ],
  "dcat:distribution": {
    "@type": "dcat:Distribution",
    "dct:conformsTo": [
      "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-distribution.jsonld"
    ]
  },
  "dcat:isVersionOf": "https://example.org/models/ieee13"
}


The load at bus 671 is written in kW and kVAr:

In [6]:
load = network.get_object('E26D83A0-D29D-41EF-9528-02C882FFCC0D')
written = next(node for node in objects if node['@id'] == f'urn:uuid:{load.uri()}')
print('written:', written['cim:EnergyConsumer.p'], 'kW,', written['cim:EnergyConsumer.q'], 'kVAr')
print('in memory:', load.p.to('kW'), 'kW,', load.q.to('kVAr'), 'kVAr')

written: 1155 kW, 660 kVAr
in memory: 1155.0 kW, 660.0 kVAr
